# **集成学习**

**原理**: 通过构建并结合多个 **“基学习器” (Base Learners)** 来完成学习任务，从而获得比单一学习器更好的预测性能。

**单一模型往往存在两个主要问题**：

- 偏差 (Bias) 高： 模型太简单，没学会数据的规律（欠拟合）。

- 方差 (Variance) 高： 模型太复杂，死记硬背了训练数据，换个数据就不行了（过拟合）。

集成学习通过组合多个模型，试图同时解决或缓解这两个问题。根据组合方式的不同，集成学习主要分为三大流派：**Bagging**、**Boosting** 和 **Stacking**。

## **Bagging(Bootstrap Aggregating，引导聚合)**

### **原理**

**核心思想**: 通过构建多个**独立且同质** 的基学习器,大家独立并行地学习,最后投票决定结果.

**构建流程**:

1. **Bootstrap（自助采样）**:从原始训练集中**有放回地**随机抽取N个样本，生成多个不同的子数据集。这个过程重复多次（比如100次），得到多个略有差异的数据子集。

2. **训练**: 用同一个基础学习算法（如决策树），在每个数据子集上独立训练一个模型。

3. **聚合结果**：

- 分类问题： 少数服从多数（投票）。

- 回归问题： 计算平均值。

**特点**: 主要目标是降低方差，特别适用于那些本身复杂度高、容易过拟合的模型（如深度决策树）。

**典型代表**： 随机森林。它是Bagging的扩展，在构建每棵决策树时，不仅对样本进行随机采样，还会随机选取一部分特征进行节点分裂。这种“双重随机性”进一步增强了模型的多样性，防止模型之间过于相似，效果通常比普通Bagging更好。

### **随机森林(Random Forest)**

#### **bootstrap(随机)采样原理**

在进入随机森林之前，先深入理解Bagging的核心——Bootstrap采样：

对于一个包含N个样本的训练集D，进行Bootstrap采样：

- 每次有放回地随机抽取1个样本

- 重复N次，形成一个包含N个样本的新训练集D_t

**数学推导：**

一个样本在单次抽取中被抽中的概率：$\frac{1}{N}$

​
一个样本在单次抽取中未被抽中的概率：$1-\frac{N-1}{N}$

​
经过N次独立抽取，一个样本从未被抽中的概率：$P(从未被抽中))=\left(1-\frac{1}{N}\right)^N$

当$N→∞$时: $\displaystyle \lim_{N \to \infty}{\left(1-\frac{1}{N}\right)^N} = e^{-1} \approx 0.368$

**重要结论**：在Bootstrap采样中，大约有**36.8%**的原始样本不会出现在采样集中，这些样本称为**袋外样本(Out-Of-Bag, OOB)**。


#### **原理**

**随机森林** 其实有两部分组成,即 **随机**和**森林**

1. **森林**: 算法内部有很多棵决策树,每棵树都是一个基学习器.


2. **随机**:这便是灵魂所在,如果训练集中有比较明显的特征,那么那么几乎所有的树都会在根节点或靠近根节点的位置使用这个特征进行分裂,导致每棵树的结构都会高度的相似.为了让每棵树都有差异性,随机森林引入了两个层面的随机性:
    1. **样本随机采样**: 每次从训练集中随机采样,构建一个新的数据集(bootstrap 采样,同bagging).
    2. **特征随机采样**: 在每个节点分裂时，不是从所有d个特征中选择最佳分裂特征，而是：
        - 先随机选取$m$个特征($m<=d$)
        - 从这$m$个特征中选择最佳分裂特征


3. **随机森林的两大“特异功能”**
   
    除了预测准确，随机森林还有两个非常有用的特性，让它在工业界备受青睐：

    **A. OOB (Out-of-Bag) 评估 —— 自带验证集**

    在 Bootstrap 抽样时，大约有 36.8% 的数据不会被抽中。

    - 这意味着，我们不需要专门切分“验证集”来测试模型好坏。

    - 我们可以直接用这 36.8% 的 OOB 数据来测试那棵树的性能。

    - 结论： 随机森林自带验证机制，省去了交叉验证的时间。

    **B. 特征重要性 (Feature Importance)** 

    随机森林可以告诉你：在这个任务中，哪个特征最重要？

    - **原理：** 如果我们把某个特征的数据打乱（加入噪声），导致树的预测错误率飙升，说明这个特征很重要；如果打乱后错误率没变化，说明这个特征可有可无。

    - **用途：** 这在数据分析和解释模型时非常有用（例如：银行想知道拒绝贷款时，是“信用记录”重要还是“收入水平”重要）。

#### **构建过程**

假设我们有一个数据集，包含 $N$ 个样本和 $M$ 个特征。随机森林的构建过程如下：

**第一步：Bootstrap 抽样（样本随机）**

我们不把所有数据都给每一棵树。对于第 $k$ 棵树：

- 我们从原始数据集中，有放回地随机抽取 $N$ 个样本。
- 结果： 有的样本可能被抽中多次，有的样本可能一次都没被抽中（Out-of-Bag / OOB (袋外样本)）。
- 目的： 保证每棵树看到的训练数据略有不同。

**第二步：特征随机选择（特征随机）**

在决策树构建过程中，通常在每个节点分裂时，会遍历所有$ M$ 个特征来找最好的分裂点。

- 但在随机森林中，**绝不遍历所有特征**。

- 它会随机选择一小部分特征（通常是 $\sqrt{M}$或 $\log_2{M}$ 个）。

- 仅在这个子集中寻找最好的分裂特征。

- 目的： 即使某个特征非常强（比如“收入”对预测“消费”很关键），我们也不希望每棵树都第一时间用它。我们要强迫一些树去关注其他次要特征，这样才能从不同角度理解数据，防止过拟合。

**第三步：构建决策树**

按照上述规则，让每棵决策树**充分生长**（通常不剪枝，或者剪枝很少）。这意味着每棵树虽然方差大（容易过拟合自己的子数据），但偏差小（学得深）。

**第四步：集成投票**

当新的数据来了，我们把它输入给森林中的每一棵树：

**分类问题**： 就像投票选举。如果是二分类，100棵树里有60棵说是A，40棵说是B，那最终结果就是 A。

**回归问题**： 计算所有树预测值的平均值。

#### **优缺点**

|优点|缺点|
|--|--|
|**准确率高：** 在大多数结构化数据上表现极佳。|**模型庞大：** 包含几百棵树，模型文件大，占用内存多。|
|**抗过拟合：** “双重随机”机制让它很难过拟合。|**预测慢：** 预测时需要几百棵树都跑一遍，比单棵树慢得多（不适合毫秒级响应的实时系统）。|
|**傻瓜式调参：** 很多时候默认参数就能跑出很好的结果。|**黑盒：** 很难解释具体的决策路径（不像单棵决策树那样可以画出清晰的流程图）。|
|**处理高维数据：** 不需要做复杂的特征筛选。||


#### **代码演示**

In [1]:
from sklearn.ensemble import RandomForestClassifier
from sklearn.datasets import load_iris
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 准备数据
data = load_iris()
X, y = data.data, data.target
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 2. 构建模型
# n_estimators=100 表示种100棵树
# max_features='sqrt' 表示每次分裂只看根号下总特征数个特征
rf = RandomForestClassifier(n_estimators=100, max_features='sqrt', random_state=42)

# 3. 训练 (并行训练，速度快)
rf.fit(X_train, y_train)

# 4. 预测
y_pred = rf.predict(X_test)

# 5. 查看结果
print(f"准确率: {accuracy_score(y_test, y_pred)}")

# 6. 查看特征重要性
print("特征重要性:", rf.feature_importances_)

准确率: 0.8888888888888888
特征重要性: [0.11106946 0.01736048 0.48712805 0.38444201]


## **Boosting(提升法)**

### **原理**


多个弱学习器**串行**执行, 每个弱学习器都在尝试改进**前一个**弱学习器的**错误**, 最终得到一个强学习器.

有两种 Boosting 算法：AdaBoost 和 GBDT。

**举个例子**: 

第一轮： 先派出一个“弱学生”（模型 A）去考试，考完后看看哪几题做错了。

第二轮： 派出第二个“弱学生”（模型 B）。注意，模型 B 的任务不是重考一遍，而是专门去攻克模型 A 做错的那些题。

第三轮： 派出模型 C，专门去攻克 A 和 B 都没搞定的难题。

最后： 把 A、B、C... 的答案加起来（加权），形成最终答案。



**关键点**：

- **串行训练**： 模型之间有依赖关系，后一个模型依赖前一个模型的结果。

- **关注错误**： 每一轮训练的目标，都是为了减小上一轮的误差。

- **弱学习器**： Boosting 通常使用很简单的模型（比如深度只有 3-5 层的决策树）作为基学习器。因为它相信通过不断的叠加，简单的模型也能变强。

### **AdaBoost(Adaptive Boosting 自适应提升)**

#### **原理**

前一个基本分类器分错的样本，会使其在后一个基本分类器中得到加强(**加权**)，加权后的全体成员组成了分类器。

AdaBoost 通常使用**决策树桩** (Decision Stump) 作为基学习器。

- 注：决策树桩就是只有一层的决策树，它像一个残废的树，一次只能切一刀，非常简单，非常弱。

#### **构建流程**

**第一步：初始化权重**

一开始，所有 $N$ 个训练样本的权重是平等的，都是 $1/N$。大家生而平等。

**第二步：迭代训练（假设循环 T 轮）**

对于每一轮 $t$ (from 1 to T):

1. **训练模型：** 使用当前权重的样本数据，训练一个弱分类器 $h_t$（比如一个树桩）。
2. **计算错误率 ($\epsilon_t$)**： 看看这个模型把多少权重的样本分错了。
3. **计算模型话语权 ($\alpha_t$)**： 这一点至关重要！
   - 如果错误率 $\epsilon_t$ 很低，说明这个模型很强，给它很高的投票权重 $\alpha_t$。
   - 如果错误率 $\epsilon_t$ 很高（接近 0.5），说明这个模型在瞎猜，给它的权重就很低。
   - 公式直觉： $$\alpha_t = \frac{1}{2} \ln(\frac{1-\epsilon_t}{\epsilon_t})$$
4. 更新样本权重： 这是最关键的一步（Adaptive 的体现）。
   - 对于分错的样本： 增加它们的权重。下一轮训练时，新的模型会优先“照顾”这些样本，因为分错它们的代价变大了。
   - 对于分对的样本： 降低它们的权重。既然已经学会了，下一轮就不用花太多精力在它们身上。


**第三步：强力结合训练完** 

T 个模型后，把它们组合起来。最终预测结果 = $sign(\alpha_1 \cdot h_1 + \alpha_2 \cdot h_2 + ... + \alpha_T \cdot h_T)$(翻译：所有弱分类器的结果，乘以它们各自的话语权，加起来看是正还是负。)

**一个具体的数学直觉**

AdaBoost 最精妙的地方在于它如何调整样本权重。

假设第 $i$ 个样本被预测错了，它的权重更新公式大概长这样：$$w_{new} = w_{old} \cdot e^{\alpha}$$假设第 $i$ 个样本被预测对了，它的权重更新公式大概长这样：$$w_{new} = w_{old} \cdot e^{-\alpha}$$因为 $\alpha$ 是正数（前提是模型准确率 > 50%），所以 $e^{\alpha} > 1$，错题权重变大。$e^{-\alpha} < 1$，对题权重变小。通过这种指数级的调整，经过几轮后，那些一直被分错的“顽固样本”权重会变得巨大，迫使算法必须要把它们分开。

#### **优缺点**

**优点**

- **精度高**： 它可以把很多个很弱的分类器组合成一个超强的分类器。

- **不容易过拟合**： 这是一个比较反直觉的结论，但在很多情况下，AdaBoost 即使训练很多轮，泛化能力依然很好（当然到了后期还是会过拟合的）。

- **简单灵活**： 它可以结合各种算法（不只是树），只要是弱学习器就行。



**缺点**

- **对异常值（Outliers）极度敏感！**

    - 为什么？ 因为 AdaBoost 的逻辑是“你越错，我越关注你”。

    - 如果是正常的“难样本”还好。但如果数据里有一个错误标注的噪音（比如把一张猫的图强行标成狗），AdaBoost 会拼命地加大这个样本的权重，发疯一样想把它分对，最后导致模型跑偏，为了这一个噪音破坏了整体规则。

- **无法并行**： 必须等前一个模型练完，算出错误率，更新完权重，才能练下一个。

#### **代码演示**

In [2]:
from sklearn.ensemble import AdaBoostClassifier
from sklearn.tree import DecisionTreeClassifier
from sklearn.datasets import make_classification
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score

# 1. 造一点数据
X, y = make_classification(n_samples=1000, n_features=20, random_state=42)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.3)

# 2. 设定基学习器：一个深度只有1的决策树（树桩），非常弱
weak_learner = DecisionTreeClassifier(max_depth=1)

# 3. 构建 AdaBoost
# n_estimators=50 : 迭代50轮
# algorithm='SAMME.R' : 一种优化的AdaBoost算法
ada = AdaBoostClassifier(estimator=weak_learner, n_estimators=50, learning_rate=1.0, random_state=42)

# 4. 训练
ada.fit(X_train, y_train)

# 5. 预测
y_pred = ada.predict(X_test)

print(f"AdaBoost 准确率: {accuracy_score(y_test, y_pred)}")

# 对比：如果我们只用那个单层的弱鸡决策树，效果如何？
weak_learner.fit(X_train, y_train)
print(f"单层决策树准确率: {weak_learner.score(X_test, y_test)}")

AdaBoost 准确率: 0.87
单层决策树准确率: 0.85


### **GBDT(Gradient Boosting Decision Tree 梯度提升决策树)**

#### **基本原理**

**残差 = 真实值 - 预测值**

**举个生动的例子：猜年龄**

假设我们要预测这名用户的年龄，他的真实年龄是 30岁。

**第一轮（第一棵树）：**
- **目标**： 预测真实年龄 30。
- **训练**： 树 A 看了看特征，给出的预测值是 20岁。
- **残差**： $30 - 20 = 10$。
- 意味着： 我们还差 10 岁没预测对。
  
**第二轮（第二棵树）**：
- **关键点来了：** 这棵树的目标不再是预测 30，而是去预测那个残差 10。
- **训练：** 树 B 看了看特征，它的预测任务是填补这 10 岁的空缺。假设它预测出了 6岁。
- **新残差：** $10 - 6 = 4$。意味着： 现在还差 4 岁。

**第三轮（第三棵树）**：
- **目标：** 去预测剩下的残差 4。
- **训练：** 树 C 很准，直接预测出了 4岁。
- **新残差：** $4 - 4 = 0$。意味着： 我们已经预测对了 4 岁。
- **最终结果（集成）**：我们将所有树的预测值加起来：$$预测值 = 树A(20) + 树B(6) + 树C(4) = 30$$
  
**这就是 GBDT 的核心思想：每一棵树都是建立在之前所有树的“错误”（残差）之上的，每一棵树都在一点点修补偏差。**


#### **为什么叫'梯度'**

**损失函数(MSE)**: $$Loss = \frac{1}{2}(y - \hat y)^2$$

如果我们对这个 Loss 求导（求梯度）：$$\frac{\partial Loss}{\partial \hat{y}} = -(y - \hat{y})$$

你会发现：**负梯度 (Negative Gradient)** 恰好就是 $y - \hat{y}$，也就是**残差**！


**GBDT 实际上是一个更广义的算法，拟合残差只是它的一种特例**